# Quick reproduction — SkillReuse (CMP263)

Reproduces the main results **without downloading the 6.5 GB raw data**, using the small derived tables committed in `data/derived/` (9.5 MB). About 5 minutes in total.

* `make quick` runs the main protocol (H1–H4, 1 repetition instead of 3) and re-runs the frozen models on the holdout (which was already evaluated once; the reproduction writes to a separate directory).
* The full pipeline (download, audit, tuning, holdout construction) is described in `README.md` and `paper/draft/07_appendix_reproducibility.md`.

In [ ]:
!pip -q install uv
!git clone https://github.com/podorize/skillreuse.git
%cd skillreuse
!uv sync

In [ ]:
!uv run skillreuse check

In [ ]:
!make quick-setup

In [ ]:
# main protocol, 1 repetition (full report uses 3): ~2 min
!uv run skillreuse primary --repeats 1 --output-dir results/owner/primary_quick

In [ ]:
# frozen models on the holdout, reproduced in a separate directory: ~2 min
!uv run skillreuse holdout-eval --output-dir results/owner/holdout_repro

In [ ]:
import json, pandas as pd
print(pd.read_csv('results/owner/primary_quick/primary_hypotheses.csv')[['hypothesis','score_a','score_b','p_holm_family']])
r = json.load(open('results/owner/holdout_repro/holdout_results.json'))
print(r['classification']['roc_auc'], r['classification']['roc_auc_ci95'])
print(r['regression']['rmsle'], r['regression']['rmsle_baseline_mean_log1p'])

## Full pipeline (optional, hours, ~7 GB of disk)

```bash
make data owner primary robustness importance final holdout
```

`make holdout-eval` is guarded by a lock (the holdout can be evaluated only once).